In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(model=model, lr=10, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo")
# opt = torch_numopt.GradientDescentLS(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="wolfe")
# opt = torch_numopt.GradientDescentLS(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="strong-wolfe")
# opt = torch_numopt.GradientDescentLS(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="goldstein")

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.21438339352607727
epoch:  1, loss: 0.13746751844882965
epoch:  2, loss: 0.09475471079349518
epoch:  3, loss: 0.07002348452806473
epoch:  4, loss: 0.055605411529541016
epoch:  5, loss: 0.04713401198387146
epoch:  6, loss: 0.04204413294792175
epoch:  7, loss: 0.038935303688049316
epoch:  8, loss: 0.03702423349022865
epoch:  9, loss: 0.035844091325998306
epoch:  10, loss: 0.03511321172118187
epoch:  11, loss: 0.03465975821018219
epoch:  12, loss: 0.034378036856651306
epoch:  13, loss: 0.03420213237404823
epoch:  14, loss: 0.03409181907773018
epoch:  15, loss: 0.034022197127342224
epoch:  16, loss: 0.03397833928465843
epoch:  17, loss: 0.033969223499298096
epoch:  18, loss: 0.03393486514687538
epoch:  19, loss: 0.03392857313156128
epoch:  20, loss: 0.03390093892812729
epoch:  21, loss: 0.03389551490545273
epoch:  22, loss: 0.0338716059923172
epoch:  23, loss: 0.033870916813611984
epoch:  24, loss: 0.033847875893116
epoch:  25, loss: 0.0338335745036602
epoch:  26, loss: 0

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7720535668095778
Test metrics:  R2 = 0.7530314446254411


In [9]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.GradientDescentLS(
    model=model,
    lr=10,
    c1=1e-4,
    tau=0.1,
    line_search_method="bisect",
    line_search_cond="gradroot"
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.23281092941761017
epoch:  1, loss: 0.032802097499370575
epoch:  2, loss: 0.032484591007232666
epoch:  3, loss: 0.0323270820081234
epoch:  4, loss: 0.03225802257657051
epoch:  5, loss: 0.031571708619594574
epoch:  6, loss: 0.030988354235887527
epoch:  7, loss: 0.030819352716207504
epoch:  8, loss: 0.030594298616051674
epoch:  9, loss: 0.03047131933271885
epoch:  10, loss: 0.030009903013706207
epoch:  11, loss: 0.02982337214052677
epoch:  12, loss: 0.029667437076568604
epoch:  13, loss: 0.029522160068154335
epoch:  14, loss: 0.028936509042978287
epoch:  15, loss: 0.02867044135928154
epoch:  16, loss: 0.028522858396172523
epoch:  17, loss: 0.02744143456220627
epoch:  18, loss: 0.026411524042487144
epoch:  19, loss: 0.026163628324866295
epoch:  20, loss: 0.025862397626042366
epoch:  21, loss: 0.025636209174990654
epoch:  22, loss: 0.02518775686621666
epoch:  23, loss: 0.02481100521981716
epoch:  24, loss: 0.024479031562805176
epoch:  25, loss: 0.024221176281571388
epoch:

In [8]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7749630846415837
Test metrics:  R2 = 0.7463169109671599
